In [ ]:
#Para leer imágenes médicas .mhd y .raw
import SimpleITK as sitk
#Para poder mostrar las imágens 
import matplotlib.pyplot as plt
#Pa la tablas csv
import pandas as pd
#Para carpetas y archivos
import os
#import tensorflow as tf
#gpus = tf.config.list_physical_devices('GPU')
#if gpus:
    #try:
        #for gpu in gpus:
            #tf.config.experimental.set_memory_growth(gpu, True)

        #print("GPU configurada correctamente")

    #except RuntimeError as e:
        #print(e)


# Cargar CT manual

In [ ]:
#Este lee tanto el .mhd y el .raw
scan = sitk.ReadImage(
 "data/subset0/1.3.6.1.4.1.14519.5.2.1.6279.6001.105756658031515062000744821260.mhd"
)
#Este convierte el CT a NumPy
image = sitk.GetArrayFromImage(scan)
#Este imprime (slices,alto,ancho)
print(image.shape)
#Este selleciona el slice 50 y lo dibuja -> con los dos comandos de abajo veo un slice pero es una sola tomografia
plt.imshow(image[50], cmap='gray')
plt.show()

# Info del Scan

In [ ]:
print(image.min())
print(image.max())
print(image.dtype)
print(image.shape[0])

# Ver anotaciones

In [ ]:
annotations = pd.read_csv("data/annotations.csv")
annotations.head()

# Buscar CT automáticamente

In [ ]:
#solo la columna seriesuid de mi csv [0] es el número de fila
seriesuid = annotations.iloc[0]['seriesuid']
found_path = None
#recorre todos mis subset para encontrar el q es
for subset in range(10):
    #construir la ruta
    folder = f"data/subset{subset}"
    #recorre los archivos
    for file in os.listdir(folder):
        #Para buscar coincidencias y decir este es 
        if file.startswith(seriesuid):
            found_path = os.path.join(folder, file)
            break
    if found_path:
        break
print(found_path)

# Abrir Scan que encontro 

In [ ]:
#Abre tomografia segun ruta de arribita
scan = sitk.ReadImage(found_path)
image = sitk.GetArrayFromImage(scan)
print(image.shape)
plt.imshow(image[50], cmap='gray')
plt.show()

In [ ]:
#Donde empieza el CT
origin = scan.GetOrigin()
print(origin)
#distancia
spacing = scan.GetSpacing()
print(spacing)

In [ ]:
coordZ = annotations.iloc[0]['coordZ']
print(coordZ)
origin = scan.GetOrigin()
spacing = scan.GetSpacing()
coordZ = annotations.iloc[0]['coordZ']
slice_index = int((coordZ - origin[2]) / spacing[2])
print(slice_index)
plt.imshow(image[slice_index], cmap='gray')
plt.show()

In [ ]:
coordX = annotations.iloc[0]['coordX']
coordY = annotations.iloc[0]['coordY']
pixelX = int((coordX - origin[0]) / spacing[0])
pixelY = int((coordY - origin[1]) / spacing[1])
print(pixelX, pixelY)
plt.figure(figsize=(8,8))
plt.imshow(image[slice_index], cmap='gray')
plt.scatter(pixelX, pixelY, color='red', s=100)
plt.show()

In [ ]:
for i in range(slice_index - 3, slice_index + 4):
    plt.figure(figsize=(5,5))
    plt.imshow(image[i], cmap='gray')
    plt.title(f"Slice {i}")
    plt.scatter(pixelX, pixelY, color='red', s=80)
    plt.show()